# NB04 — Reading Computational Biology Benchmarks

**Module:** 19 — Research Seminar and Paper Reading  
**Estimated time:** 2 hours  

---

## Motivation

Computational biology papers regularly introduce new tools by benchmarking them against prior work. A benchmark appears objective — numbers, rankings, leaderboards — but it is among the easiest things to misuse, accidentally or deliberately. Learning to evaluate a benchmark is essential for deciding which tools are actually worth using and which results are trustworthy.

## 1. What Makes a Valid Computational Biology Benchmark?

A valid benchmark must satisfy all of the following:

| Criterion | Valid | Invalid |
|-----------|-------|--------|
| **Data independence** | Test data held out from development | Evaluated on training data or on data the tool was tuned to |
| **Baselines** | Includes the strongest known methods | Compares only to weak or obsolete baselines |
| **Metric consistency** | Same metric(s) applied to all methods | Different metrics for different methods |
| **Dataset diversity** | Multiple datasets with different properties | Single dataset |
| **Fair configuration** | All methods run with their best known parameters | Competitors run with default (possibly suboptimal) params |
| **Statistical testing** | Differences between methods tested statistically | "Our method scores X, theirs scores X-0.01" without test |
| **Code and data** | Full benchmark reproducible from a repository | Results reported without runnable code |
| **Author independence** | Neutral benchmark not designed by the method's authors | Tool authors run the benchmark that evaluates their own tool |

## 2. Common Benchmark Flaws

1. **Evaluation on training data:** the method was tuned (or implicitly trained) on the benchmark data — reported performance is optimistic.
2. **Cherry-picked metrics:** a new method reports only the metric it wins; omits metrics where baselines win.
3. **Weak baselines:** no comparison to the current state-of-the-art (usually because the current SOTA wins).
4. **Single dataset:** a method that wins on one dataset may lose on the next. One dataset tells you nothing about generalization.
5. **Unfair parameter tuning:** the new method is tuned to the benchmark; baselines are run with defaults.
6. **Overlapping datasets:** if the training and test sets share samples (batch effects not accounted for in scRNA-seq), performance is inflated.
7. **Missing uncertainty:** only reporting means, not confidence intervals or variability across runs.

## 3. Python Implementation: Benchmark Scoring Rubric

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Optional


@dataclass
class BenchmarkRubric:
    """Score a benchmark paper on 8 validity criteria (0-10 total scale)."""

    data_independence: bool        # Test data held out from development
    strong_baselines: bool         # Strongest known methods included
    metric_consistency: bool       # Same metrics for all methods
    dataset_diversity: bool        # Multiple diverse datasets
    fair_configuration: bool       # Competitors run with best parameters
    statistical_testing: bool      # Differences tested statistically
    reproducible_code: bool        # Full runnable code provided
    author_independence: bool      # Benchmark not run solely by method authors

    notes: Optional[str] = None

    # Weights: some criteria matter more than others
    WEIGHTS = {
        'data_independence': 2.0,
        'strong_baselines': 2.0,
        'metric_consistency': 1.5,
        'dataset_diversity': 1.5,
        'fair_configuration': 1.0,
        'statistical_testing': 1.0,
        'reproducible_code': 0.5,
        'author_independence': 0.5,
    }

    def score(self) -> float:
        """Return weighted score out of 10."""
        total_weight = sum(self.WEIGHTS.values())
        earned = sum(
            self.WEIGHTS[k] for k, v in self.__dict__.items()
            if k in self.WEIGHTS and v is True
        )
        return round(10 * earned / total_weight, 1)

    def grade(self) -> str:
        s = self.score()
        if s >= 8.5:
            return 'Excellent'
        elif s >= 7.0:
            return 'Good'
        elif s >= 5.0:
            return 'Acceptable'
        elif s >= 3.0:
            return 'Weak'
        else:
            return 'Unreliable'

    def failed_criteria(self) -> List[str]:
        return [k.replace('_', ' ') for k, v in self.__dict__.items()
                if k in self.WEIGHTS and v is False]


# Three synthetic benchmark scenarios
benchmark_fair = BenchmarkRubric(
    data_independence=True,
    strong_baselines=True,
    metric_consistency=True,
    dataset_diversity=True,
    fair_configuration=True,
    statistical_testing=True,
    reproducible_code=True,
    author_independence=False,  # authors ran their own benchmark
    notes='Gold-standard benchmark — all criteria met except independence'
)

benchmark_typical = BenchmarkRubric(
    data_independence=True,
    strong_baselines=False,  # missing 2 recent baselines
    metric_consistency=True,
    dataset_diversity=False,  # single dataset
    fair_configuration=False,  # baselines run with defaults
    statistical_testing=False,
    reproducible_code=True,
    author_independence=False,
    notes='Typical tool paper benchmark — passes basics, fails on rigor'
)

benchmark_biased = BenchmarkRubric(
    data_independence=False,  # evaluated on data used for parameter selection
    strong_baselines=False,
    metric_consistency=False,  # reports AUROC for new method, accuracy for baselines
    dataset_diversity=False,
    fair_configuration=False,
    statistical_testing=False,
    reproducible_code=False,
    author_independence=False,
    notes='Red-flag benchmark — should not be trusted'
)

for label, bm in [('Fair', benchmark_fair), ('Typical', benchmark_typical), ('Biased', benchmark_biased)]:
    print(f"{label}: score={bm.score()}/10, grade='{bm.grade()}'")
    if bm.failed_criteria():
        print(f"  Failed: {', '.join(bm.failed_criteria())}")

In [ ]:
# Simulate fair vs biased benchmark results
np.random.seed(42)

def simulate_benchmark(n_methods: int = 5, n_datasets: int = 4,
                        biased: bool = False) -> dict:
    """Simulate benchmark performance across methods and datasets.

    In biased mode: the first method (the 'proposed' one) gets a tuning advantage.
    """
    methods = ['Proposed'] + [f'Baseline_{i}' for i in range(1, n_methods)]
    datasets = [f'Dataset_{chr(65+i)}' for i in range(n_datasets)]

    # True underlying performance (AUROC, higher is better)
    true_perf = np.random.uniform(0.70, 0.85, n_methods)
    true_perf[0] = 0.78  # Proposed method is actually middle-of-the-pack

    results = np.zeros((n_methods, n_datasets))
    for m in range(n_methods):
        for d in range(n_datasets):
            noise = np.random.normal(0, 0.02)
            if biased and m == 0:
                # Proposed method was tuned on all datasets — inflated performance
                results[m, d] = min(true_perf[m] + 0.05 + noise, 1.0)
            else:
                results[m, d] = np.clip(true_perf[m] + noise, 0, 1)

    return {
        'methods': methods,
        'datasets': datasets,
        'results': results,
        'mean_performance': results.mean(axis=1),
    }


fair_bm = simulate_benchmark(biased=False)
biased_bm = simulate_benchmark(biased=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Benchmark Validity — Fair vs Biased Evaluation', fontsize=13, fontweight='bold')

# Panel 1: Benchmark validity scorecard (bar chart)
ax1 = axes[0]
scores = [benchmark_fair.score(), benchmark_typical.score(), benchmark_biased.score()]
colors = ['#43A047', '#FB8C00', '#E53935']
labels = ['Fair', 'Typical', 'Biased']
bars = ax1.barh(labels, scores, color=colors, alpha=0.85, height=0.5)
ax1.set_xlim(0, 10)
ax1.axvline(7, color='gray', linestyle='--', alpha=0.6, label='Good threshold (7)')
ax1.set_xlabel('Validity score (0–10)')
ax1.set_title('Benchmark Validity Scores')
for bar, score in zip(bars, scores):
    ax1.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{score}', va='center', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, axis='x', alpha=0.3)

# Panel 2: Fair benchmark leaderboard across datasets
ax2 = axes[1]
x = np.arange(len(fair_bm['datasets']))
width = 0.15
colors_methods = plt.cm.Set2(np.linspace(0, 1, len(fair_bm['methods'])))
for i, (method, color) in enumerate(zip(fair_bm['methods'], colors_methods)):
    offset = (i - len(fair_bm['methods'])/2) * width
    ax2.bar(x + offset, fair_bm['results'][i], width,
            label=method, color=color, alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(fair_bm['datasets'])
ax2.set_ylabel('AUROC')
ax2.set_title('Fair Benchmark: Performance\nAcross Datasets')
ax2.set_ylim(0.6, 1.0)
ax2.legend(fontsize=8)
ax2.grid(True, axis='y', alpha=0.3)

# Panel 3: Fair vs biased — mean performance comparison
ax3 = axes[2]
x_pos = np.arange(len(fair_bm['methods']))
ax3.bar(x_pos - 0.2, fair_bm['mean_performance'], 0.35,
        label='Fair evaluation', color='steelblue', alpha=0.85)
ax3.bar(x_pos + 0.2, biased_bm['mean_performance'], 0.35,
        label='Biased evaluation', color='salmon', alpha=0.85)
ax3.set_xticks(x_pos)
ax3.set_xticklabels(fair_bm['methods'], rotation=30, ha='right')
ax3.set_ylabel('Mean AUROC')
ax3.set_title('Fair vs Biased Benchmark:\nProposed Method Inflation')
ax3.set_ylim(0.6, 1.0)
ax3.legend()
ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('benchmark_validity.png', dpi=150, bbox_inches='tight')
plt.show()

## Exercises

**Exercise 1:** Find the methods section of Mangul et al. (2019) "Systematic benchmarking of omics computational tools." Score it using `BenchmarkRubric`. Which criteria does it satisfy?

**Exercise 2:** A paper introduces a new scRNA-seq clustering tool and benchmarks it on three datasets, reporting ARI (Adjusted Rand Index) only. It does not include Seurat or Scanpy as baselines. Score this benchmark. What is the most significant flaw?

**Exercise 3:** Explain in one sentence why a biased benchmark can still have peer-reviewed publication status. (Hint: what does peer review actually check?)

**Exercise 4:** Modify the `simulate_benchmark` function to model the scenario where the proposed method was tuned specifically on Dataset_A but not on B, C, D. What does the performance profile look like?

## Quiz

1. Name three criteria a valid benchmark must satisfy.
2. What is cherry-picking metrics, and give a concrete example from computational biology?
3. Why does a single-dataset benchmark fail to support generalization claims?
4. What is "fair configuration" and why does it matter?
5. If you see a paper where the authors ran their own benchmark, should you immediately distrust it? What should you check instead?

## References

- Mangul et al. (2019). Systematic benchmarking of omics computational tools. Nature Comm 10:1393.
- Boulesteix et al. (2013). A plea for neutral comparison studies. PLOS ONE 8(4):e61562.
- Teschendorff, A.E. (2021). Avoiding common pitfalls in machine learning omic data science. Nature Materials 20:1232–1239.